In [114]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.colors import to_rgb, to_hex
from collections import defaultdict
import os

In [ ]:
outdir = Path("./plots/")
outdir.mkdir(parents=True, exist_ok=True)

# define constants
colname = "assigned_subcat" # hierarchy level to be analyzed, from here on defined as 'subcategory'
colname_map = "Subcategory" # respective column name in the mapping file
top_colname = "assigned_top" # hierarchy level to serve as top-level category/mapping
top_colname_map = "Top_Category" # respective column name in the mapping file
seq_id = 0.95 # sequence identity threshold used for clustering (part of input data file name)

# assertions
assert colname and top_colname in ["assigned_subcat", "assigned_cat", "assigned_top"], "Invalid column name. Must be one of 'assigned_subcat', 'assigned_cat', or 'assigned_top'."
assert colname_map and top_colname_map in ["Subcategory", "Category", "Top_Category"], "Invalid column name. Must be one of 'Subcategory', 'Category', or 'Top_Category'."
assert seq_id in [0.95, 0.2], "Invalid sequence identity threshold. Must be one of 0.95, 0.2."

# read mapping file for keywords to sub-, mid-, and top-level categories
keywords_map = pd.read_csv("./data/Category_map.csv") 

def clean_df(df, colname):
    """"Helper function to clean the dataframe by replacing entries with semicolons in the specified column with 'Unknown'."""
    mask = df[colname].str.contains(";")
    df.loc[mask, colname] = "Unknown"
    return df

# read in dataframe and clean it
all_repr = pd.read_csv(f"./data/all_new_representatives_assigned_cat_{seq_id}.tsv", sep="\t") # all representatives, including phages, bacteria, and viruses
all_repr = clean_df(all_repr, colname)

In [116]:
# get counts of subcategories, excluding "Unknown"
data_counts = all_repr[colname].value_counts()
data_known_counts = data_counts[(data_counts.index != "Unknown")]

def counts_to_df(vc):
    """Helper function to convert a value counts series to a dataframe."""
    return vc.rename("count").reset_index().rename(columns={"index": colname})

# convert counts to dataframe
df_counts = counts_to_df(data_counts)

# map subcategories to top-level categories
if colname != "assigned_top":
    df_counts[top_colname] = df_counts[colname].map(CATEGORY_TO_TOP)
else: 
    df_counts[top_colname] = df_counts[colname]

# save
df_counts.to_csv(os.path.join(outdir, f"{colname}_counts.tsv"), sep="\t", index=False)

# exclude Unknowns for further analysis
df_counts = df_counts[df_counts[colname] != "Unknown"]

In [117]:
### plotting utilities 

# define colors for top-level categories
TOPCAT_COLORS = {
    "Assembly": "#007BFF",
    "Structural": "#1e00ff",
    "Host takeover": "#2fff00",
    "NA processing": "#00ffb7",
    "Replication":"#00d9ff",
    "Translation": "#f6ff00",
    "AMGs": "#ffbf00", #"#ffa200",
    "Lysis": "#ff0000", 
    "Anti-Defense": "#ff00dd", 
    "Effectors": "#8e01f9", 
    "Lysogeny": "#FF5A00", #"#ff007b",
    "Unknown": "#bdbdbd", 
}

# define colors for mid-level categories
CAT_COLORS = {
    "Transcriptional regulation": "#2fff00",
    "Transcriptional inhibition": "#02d136",
    "Host resource takeover": "#39fa69",
    "Transcriptional antitermination": "#60c006",
    "Host takeover via phosphorylation": "#3ea52e", 
    "Lytic switch": "#74d216",
    "Transcription factor": "#3C6D0F",
    "Transcriptional machinery": "#0D6806",

    "Integration": "#00ffb7",
    "Excision and transcriptional regulation": "#0dd09c",
    "DNA repair": "#0D916D",
    "Recombination": "#036D51",
    "RNA modification": "#71F8D4",
    "RNA cleavage": "#37F9AF",
    "RNA sealing": "#14AC72",

    "Maintenance": "#FF5A00", #"#ff007b",

    "Protein synthesis": "#f4fa51",
    "Ribosomal protein": "#f3fb00",
    "Translation regulation": "#d0d703",
    "Ribosome biogenesis": "#a0a512",
    "Protein maturation and folding": "#FFFF00",

    "Head morphogenesis": "#4099F9",
    "Head-to-tail joining": "#74B7FE",
    "Tail assembly": "#0F559F",
    "Baseplate assembly": "#014894",
    "DNA packaging": "#0E437B",

    "Replication initiation":"#00d9ff",
    "DNA replication":"#3AAEC3",

    "Capsid": "#1e00ff",
    "DNA injection": "#1903c1",
    "Tail and Fibers": "#553ffb",
    "Baseplate": "#3c2dad",

    "Host polysaccharide degradation": "#e24141", 
    "Endolysin": "#ff0000", 
    "Spanin": "#a90202", 
    "Lysis regulation": "#7D1C1C", 

    "Anti-Restriction": "#ff00dd", 
    "Anti-TA": "#e634f6", 
    "Anti-Defense": "#df00fd",

    "Nucleotide metabolism": "#ffbf00",
    "Other AMGs": "#ff9100",

    "Virulence factors": "#af01f9", 
    "Stress response and damage protection": "#7d01f9", 
    "Membrane transport and signaling": "#a312c3", 
    "Superinfection exclusion": "#710589", 

    "Unknown": "#bdbdbd", 
}


def adjust_color(color, factor=0.2, lighten=True):
    """Mix a color with white or black. The closer the value to 1, the more white/black contained"""
    r, g, b = to_rgb(color)

    if lighten:
        r = r + (1-r)*factor
        g = g + (1-g)*factor
        b = b + (1-b)*factor
    else:
        r = r * (1-factor)
        g = g * (1-factor)
        b = b * (1-factor)

    return to_hex((r, g, b))


def get_factor_range(n, min_n=2, max_n=20, min_spread=0.1, max_spread=0.5):
    """Returns factor range depending on n"""
    # normalize n to 0-1
    t = min(1, max(0, (n - min_n) / (max_n - min_n)))

    spread = min_spread + t * (max_spread - min_spread)

    base = 0.15 
    return base, base+spread


def generate_subcategory_colors(TOPCAT_COLORS, CATEGORY_TO_TOP):
    """Generate colors for subcategories based on their top-level category colors."""
    top_to_sub = defaultdict(list)
    for sub, top in CATEGORY_TO_TOP.items():
        top_to_sub[top].append(sub)

    sub_colors = {}

    for top, subs in top_to_sub.items():
        base_color = TOPCAT_COLORS.get(top, "#999999")
        n = len(subs)

        low, high = get_factor_range(n)

        if n == 1:
            factors = [0.2]
        else:
            factors = np.linspace(low, high, n)

        for i, (sub, factor) in enumerate(zip(subs, factors)):
            lighten = (i % 2 == 0) # alternate
            sub_colors[sub] = adjust_color(base_color, factor, lighten)

    return sub_colors

# lighten top-level category colors
TOPCAT_COLORS_ADJUSTED = {
    k: adjust_color(v, factor=0.2, lighten=True)
    for k, v in TOPCAT_COLORS.items()
}

In [118]:
if colname == "assigned_top":
    plt.figure(figsize=(5, 3))

    sns.barplot(data=df_counts, x=colname, y="count", palette=TOPCAT_COLORS_ADJUSTED, hue=colname, legend=False, errorbar=None, edgecolor="black")

    # customize axes
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.xlabel("")

    for p in ax.patches:
        height = p.get_height()

        if height == 0:
            continue

        x = p.get_x() + p.get_width() / 2

        ax.text(x, height, f"{int(height)}", ha="center", va="bottom", fontsize=7)

    ymax = df_counts["count"].max()
    ax.set_ylim(0, ymax * 1.08)

    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{colname}_counts.png"), dpi=300, bbox_inches="tight")
    plt.close()

else:
    if top_colname == "assigned_cat":
        coloring = CAT_COLORS

    if top_colname == "assigned_top":
        coloring = TOPCAT_COLORS

    # create a mapping from subcategories to top-level categories
    CATEGORY_TO_TOP = keywords_map[[colname_map, top_colname_map]].drop_duplicates().set_index(colname_map)[top_colname_map].to_dict()
    
    # generate colors for subcategories based on their top-level category colors
    SUBCATEGORY_COLORS = generate_subcategory_colors(coloring, CATEGORY_TO_TOP)  

    facet_order = ["NA processing", "Host takeover", "Assembly", "Structural", "Replication", "Lysis", "Anti-Defense", "Lysogeny", "AMGs", "Effectors", "Translation"]

    df_counts[top_colname] = pd.Categorical(df_counts[top_colname], categories=facet_order, ordered=True)

    g = sns.FacetGrid(df_counts, col=top_colname, col_wrap=3, height=4, aspect=1.5, sharey=False, sharex=False, col_order=facet_order)

    g.map_dataframe(sns.barplot, x=colname, y="count", hue=colname, dodge=False, errorbar=None, edgecolor="black", palette=SUBCATEGORY_COLORS, legend=False)

    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.title.set_fontsize(16)
        ax.set_xticks(range(len(ax.get_xticklabels())))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.tick_params(axis="x", labelsize=12)
        ax.set_ylabel("Count")
        ax.set_xlabel("")

        if colname == "assigned_subcat":
            for container in ax.containers:
                ax.bar_label(container, fmt='%d', padding=2, fontsize=8)
        
        else: 
            for p in ax.patches:
                height = p.get_height()
                if height == 0:
                    continue

                x = p.get_x() + p.get_width() / 2

                # get corresponding label (subcategory name)
                label = ax.get_xticklabels()[int(round(p.get_x()))].get_text()

                if label == "Maintenance":
                    # inside the bar
                    ax.text(x, height * 0.5, f"{int(height)}", ha="center", va="center", fontsize=15, color="black")
                else:
                    # above the bar
                    ax.text(x, height, f"{int(height)}", ha="center", va="bottom", fontsize=15)
        
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1)

    g.figure.tight_layout()
    plt.savefig(os.path.join(outdir, f"{colname}_counts.png"), dpi=300, bbox_inches="tight")
    plt.close(g.figure)